# Weka gateway benchmarking

Generate output JSON that we can then aggregate, plot, and do statistics on.

## Methodology

This test uses the Butler to get handles for a large number of queries from one or more datasets.  Acquiring the query handles does not pull the actual query result data.  We then select a much smaller sample of those queries to retrieve the result data.

For each query within a given dataset, we measure the size of the HTTP object (referenced with a signed URL via the Weka-S3 gateway at USDF) to retrieve, and fetch the object.  The fetch is performed with `curl`, because we believe that `curl` redirected to `/dev/null` will be much faster than pulling an HTTP object with a Python library.

After all objects have been fetched, the retrieval is (optionally) repeated, to give us insight into the cache size and replacement policy.

This process is then repeated for each dataset.

Statistics (mean, median, population variance, population standard deviation, and coefficient of variation) are collected for each of: object size, initial pull speed, repeated pull speed, and ratio of re-checked speed to initial speed.

This is then persisted into a file in a shared directory, so we can eventually aggregate over those files, allowing us to test performance of many retrievals in parallel via the `mobu` testing framework.

In [1]:
# Imports
import json
from pathlib import Path

from weka_helpers import (
    Dataset,
    PrimitiveResultMap,
    convert_results_to_primitive,
    write_results
)

## Set up input parameters

In [2]:
# Set up report output directory
output_dir=Path("/rubin/bench/weka")  # Only on data-int

# Butler setup
butler_config="dp2"
butler_collections="dp2"

# Unset this to *not* repeat the test with the same files
#  (to test whether Weka caching affects speed)
recheck_speeds = True

# Define our input data parameters

data_map: list[Dataset] = [ 
    Dataset(
        display_name = "datasets",
        dataset_name = "visit_image",  # 30 MB each-ish
        count = 3,
        limit = 10_000,
    ),
    Dataset(
        display_name = "big_datasets",
        dataset_name = "run_provenance",  # 1GB each-ish, variable
        count = 0,
        limit = 10_000,
    ),
]

# anything else with a "+" on this page is also hosted at USDF: https://rubinobs.atlassian.net/wiki/x/CgCveg
# (anything without a +/pencil is on GCS)

## Fetch results

In [3]:
res=write_results(
    butler_config=butler_config,
    butler_collections=butler_collections,
    data_map=data_map,
    output_dir=output_dir,
    recheck_speeds=recheck_speeds)

print(json.dumps(convert_results_to_primitive(res),sort_keys=True,indent=2))

Butler query of datasets(visit_image), limit=10000...sampling 3 records
Getting size for datasets: 1/3...24719040 bytes
Getting size for datasets: 2/3...22230720 bytes
Getting size for datasets: 3/3...23137920 bytes
Getting initial speed for datasets: 1/3...10590617.6 bytes/s
Getting initial speed for datasets: 2/3...19084083.2 bytes/s
Getting initial speed for datasets: 3/3...11219763.2 bytes/s
Getting recheck speed for datasets: 1/3...19818086.4 bytes/s
Getting recheck speed for datasets: 2/3...18454937.6 bytes/s
Getting recheck speed for datasets: 3/3...14680064.0 bytes/s
[
  {
    "count": 3,
    "dataset_name": "visit_image",
    "display_name": "datasets",
    "initial_statistics_measured": {
      "CV": 0.2772424584739783,
      "mean": 13444046.16754958,
      "median": 11087395.603073891,
      "pstdev": 3727260.411329112,
      "pvariance": 13892470173861.26
    },
    "initial_statistics_reported": {
      "CV": 0.2834696272721274,
      "mean": 13631488.0,
      "median": 1